# ToolPool: Parallel Execution Across Devices

Demo notebook for the ToolPool system. Walks through automatic parallel fan-out, cost-aware scheduling, and multi-GPU tool execution.

**Key idea**: ToolPool intercepts `@tool`-decorated calls transparently, partitions iterable inputs across GPUs, runs them in parallel, and reassembles results in original order. No changes to tool code needed.

In [1]:
from time import time

from bio_programming_tools.tools.structure_prediction.esmfold import (
    ESMFoldInput,
    run_esmfold,
)
from bio_programming_tools.utils import display_gpu_memory_usage
from bio_programming_tools.utils.device_manager import DeviceManager
from bio_programming_tools.utils.tool_instance import ToolInstance
from bio_programming_tools.utils.tool_pool import ToolPool

In [2]:
# Base fragments — vary lengths so LPT scheduling has something to work with
_FRAGMENTS = [
    # ~15 residues (short)
    "MKTLLILAVVAAALA",
    "GAVLTVLLGGLLLA",
    "MGQQPGKVLGDQRR",
    "ASTVKFLGPVLDAA",
    # ~75 residues (medium)
    "MKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTG",
    "GAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSD",
    "MGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNP",
    "ASTVKFLGPVLDAAFLGSRVTSGPWKLSFPYELLHQALSGARVACFGCGDSSWEQFAVDYAALQMFRQFSMRYVAK",
    # ~150 residues (long)
    "MKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTGMKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTG",
    "GAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSDGAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSD",
    "MGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNPMGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNP",
    "ASTVKFLGPVLDAAFLGSRVTSGPWKLSFPYELLHQALSGARVACFGCGDSSWEQFAVDYAALQMFRQFSMRYVAKGAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSD",
    # ~300 residues (extra long)
    "MKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTGGAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSDMGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNPASTVKFLGPVLDAAFLGSRVTSGPWKLSFPYELLHQALSGARVACFGCGDSSWEQFAVDYAALQMFRQFSMRYVAK",
    "GAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSDMKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTGASTVKFLGPVLDAAFLGSRVTSGPWKLSFPYELLHQALSGARVACFGCGDSSWEQFAVDYAALQMFRQFSMRYVAK",
    "MGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNPMKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTGGAVLTVLLGGLLASQRANPSLFKTQLHFATATRDITYGAWVSRQFNAPNGESYEFHISINAQTQGKAIFSIQGDSD",
    "ASTVKFLGPVLDAAFLGSRVTSGPWKLSFPYELLHQALSGARVACFGCGDSSWEQFAVDYAALQMFRQFSMRYVAKMGQQPGKVLGDQRRPSLALVQRSGLYLEELFRPYIPREDLAGHQLDMLDRSGKGPGLRLYVLAAFLFAKQVLAQNPMKTLLILAVVAAALASPAGADYAKFEVTQSALTGTENSVDQKTYTFTIDNAPDDKSGKATFTVNITNPFSTQAWTG",
]

# 3x replication with single-residue suffixes to make each copy unique
# (avoids @tool dedup so the benchmark measures pure parallelism)
SEQUENCES = (
    [s + "A" for s in _FRAGMENTS]   # batch 1
    + [s + "G" for s in _FRAGMENTS] # batch 2
    + [s + "V" for s in _FRAGMENTS] # batch 3
)

print(f"{len(SEQUENCES)} sequences (all unique)")
print(f"Length distribution: {sorted(set(len(s) for s in SEQUENCES))}")
print(f"Total residues: {sum(len(s) for s in SEQUENCES):,}")

48 sequences (all unique)
Length distribution: [15, 16, 77, 153, 229, 305]
Total residues: 5,919


# Summary

ToolPool automatically parallelizes tool calls across multiple GPUs. No changes to tool code — just wrap your calls in a `ToolPool()` context:

- **Transparent interception** — any tool with an iterable input field is automatically partitioned and parallelized
- **Cost-aware scheduling** — items are distributed using LPT bin-packing based on `item_cost()` estimates
- **Built-in persistence** — workers stay alive across calls within the pool, avoiding model reload overhead
- **Device auto-detection** — discovers all visible GPUs automatically, or accepts an explicit device list
- **Automatic dedup** — duplicate items in the input are computed once and expanded back to all positions

---
## 1. Basic Usage: Auto-Detect GPUs

The simplest way to use ToolPool — just wrap your tool calls. ToolPool auto-detects all visible GPUs and partitions the input across them.

Both runs below use **warm** workers (model already loaded) so we're measuring pure inference throughput, not cold-start overhead. The warmup cell below preloads ESMFold on all GPUs first.

In [3]:
complexes = [[seq] for seq in SEQUENCES]
warmup_input = ESMFoldInput(complexes=[["MKTL"]])

# --- Sequential: single GPU, warm worker ---
print("=== Sequential (1 GPU, warm) ===")
with ToolInstance.persist():
    t0 = time()
    result_seq = run_esmfold(ESMFoldInput(complexes=complexes))
    sequential_time = time() - t0
    print(f"Time: {sequential_time:.1f}s for {len(complexes)} structures")
    display_gpu_memory_usage()

print("cleared")
display_gpu_memory_usage()

# --- Parallel: split across all GPUs, warm workers ---
print("\n=== Parallel (ToolPool, auto-detect GPUs) ===")
with ToolPool():
    t0 = time()
    result_par = run_esmfold(ESMFoldInput(complexes=complexes))
    parallel_time = time() - t0
    print(f"Time: {parallel_time:.1f}s for {len(complexes)} structures")
    display_gpu_memory_usage()

print(f"\nSpeedup: {sequential_time / parallel_time:.2f}x")
print(f"Structures returned in order: {len(result_par.structures)} (same as input)")

# GPU memory cleared
print("\n(cleaned up)")
display_gpu_memory_usage()

=== Sequential (1 GPU, warm) ===


Folding structures (ESMFold): 100%|██████████| 48/48 [01:14<00:00,  1.55s/structure]


Time: 75.2s for 48 structures
GPU 0:  29.4 /  85.5 GB  ████░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
cleared
GPU 0:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== Parallel (ToolPool, auto-detect GPUs) ===


Folding structures (ESMFold): 100%|██████████| 26/26 [00:32<00:00,  1.26s/structure]


Time: 33.0s for 48 structures
GPU 0:  20.7 /  85.5 GB  ███░░░░░░░░░░░
GPU 1:  21.6 /  85.5 GB  ███░░░░░░░░░░░

Speedup: 2.28x
Structures returned in order: 48 (same as input)

(cleaned up)
GPU 0:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░


---
## 2. Persistence Across Calls

Workers stay alive across calls within the same ToolPool context. The first call pays the model-loading cost; subsequent calls run at full speed.

In [4]:
complexes = [[seq] for seq in SEQUENCES]

print("=== Persistence demo: cold vs warm calls ===")
with ToolPool(devices=["cuda:0", "cuda:1"]):
    # Cold call — pays model loading on both GPUs
    t0 = time()
    result1 = run_esmfold(ESMFoldInput(complexes=complexes))
    cold_time = time() - t0
    print(f"Cold call: {cold_time:.1f}s for {len(complexes)} structures")
    display_gpu_memory_usage()

    # Warm call — workers already loaded, pure inference
    t0 = time()
    result2 = run_esmfold(ESMFoldInput(complexes=complexes))
    warm_time = time() - t0
    print(f"\nWarm call: {warm_time:.1f}s for {len(complexes)} structures")
    display_gpu_memory_usage()

    # Another warm call to confirm consistency
    t0 = time()
    result3 = run_esmfold(ESMFoldInput(complexes=complexes))
    warm_time2 = time() - t0
    print(f"\nWarm call 2: {warm_time2:.1f}s for {len(complexes)} structures")

print(f"\nCold → Warm speedup: {cold_time / warm_time:.2f}x")
print(f"Model loading overhead: ~{cold_time - warm_time:.0f}s")

print("\n(cleaned up)")
ToolInstance.clear_all()
DeviceManager.reset_instance()
display_gpu_memory_usage()

=== Persistence demo: cold vs warm calls ===


Folding structures (ESMFold): 100%|██████████| 26/26 [00:33<00:00,  1.29s/structure]


Cold call: 33.8s for 48 structures
GPU 0:  20.7 /  85.5 GB  ███░░░░░░░░░░░
GPU 1:  21.6 /  85.5 GB  ███░░░░░░░░░░░


Folding structures (ESMFold): 100%|██████████| 26/26 [00:18<00:00,  1.40structure/s]



Warm call: 18.9s for 48 structures
GPU 0:  20.7 /  85.5 GB  ███░░░░░░░░░░░
GPU 1:  21.6 /  85.5 GB  ███░░░░░░░░░░░


Folding structures (ESMFold): 100%|██████████| 26/26 [00:18<00:00,  1.40structure/s]



Warm call 2: 18.9s for 48 structures

Cold → Warm speedup: 1.79x
Model loading overhead: ~15s

(cleaned up)
GPU 0:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
